# Command extraction debug

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.signal import savgol_filter
import ipywidgets as widgets
from IPython.display import display, clear_output

plt.style.use('ggplot')

PKG_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PKG_ROOT))

from pipeline.config import load_config
from pipeline.extraction import extract_commands
from pipeline.frame import merge_adsb_modes, to_node_fdm_frame
from pipeline.opendata import route_dataset_dir

ROUTE_NAME = 'EHAM_LSZH'
ROUTE_DIR = route_dataset_dir(ROUTE_NAME)
ADSB_DIR = ROUTE_DIR / 'data' / 'adsb'
MODES_DIR = ROUTE_DIR / 'data' / 'modes_decoded'

manifest = pd.read_parquet(ROUTE_DIR / 'manifest.parquet')
FLIGHT_IDS = manifest.loc[manifest['status'] == 'done', 'flight_id'].astype(str).tolist()
print(f'{len(FLIGHT_IDS)} flights | route {ROUTE_DIR.name}')

300 flights | route EHAM_LSZH


## CONFIG

In [2]:
CONFIG = load_config()
display(pd.DataFrame(CONFIG).T)

,mode,tol,min_len,alt_threshold,use_alt,smooth_window,smooth_method,min_abs_value,fill,sigma_s,sigma_r,n_passes,tol_ftmin,min_alt_ft,bridge_s
mach,savgol_mach,0.01,120,20000,True,200,binned,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
cas,savgol_cas,2.5,60,NaN,False,150,binned,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
vz,savgol_vz,25,15,NaN,False,51,savgol,-1,"{'enabled': True, 'bridge_gaps': True, 'max_ga...",NaN,NaN,NaN,NaN,NaN,NaN
h_sel,bilateral_vz,NaN,10,NaN,NaN,NaN,NaN,NaN,NaN,4,250,2,250,NaN,NaN
mach_regime,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,28000.0,600.0
add_alt,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False


## 1) Load observed (1 Hz grid)

In [3]:
def load_observed(flight_id: str) -> pd.DataFrame:
    adsb = pd.read_parquet(ADSB_DIR / f'{flight_id}.parquet')
    modes = pd.read_parquet(MODES_DIR / f'{flight_id}.parquet')
    out = to_node_fdm_frame(merge_adsb_modes(adsb, modes)).copy()
    out.loc[:, 't_s'] = (out['timestamp'] - out['timestamp'].iloc[0]).dt.total_seconds()
    return out

## 2) Plateau detector

In [4]:
def infer_commands(df: pd.DataFrame, cfg: dict) -> pd.DataFrame:
    return extract_commands(df, cfg)

## 3) Browse flights

In [5]:
FLIGHT_INDEX = globals().get('FLIGHT_INDEX', 0)
flight_id = FLIGHT_IDS[FLIGHT_INDEX]

In [6]:
def plot_flight(d, title):
    t = d['t_s']
    fig = make_subplots(
        rows=6,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.03,
        subplot_titles=('Altitude', 'Vertical rate', 'CAS', 'Mach', 'TAS integration', 'Gamma integration'),
    )

    fig.add_trace(go.Scatter(x=t, y=d['altitude'], mode='lines', line=dict(color='gray', width=1), name='observed altitude'), row=1, col=1)
    fig.add_trace(go.Scatter(x=t, y=d['h_sel'], mode='lines', line=dict(color='red', width=2, shape='hv'), name='h_sel'), row=1, col=1)
    fig.add_trace(go.Scatter(x=t, y=d['fdm_alt_sel_ft'], mode='lines', line=dict(color='royalblue', width=1.4), name='v2 alt plateau'), row=1, col=1)

    fig.add_trace(go.Scatter(x=t, y=d['vertical_rate'], mode='lines', line=dict(color='gray', width=1), name='observed VS'), row=2, col=1)
    m = d['vz_sel'].notna()
    fig.add_trace(go.Scatter(x=t[m], y=d.loc[m, 'vz_sel'], mode='lines', line=dict(color='red', width=2, shape='hv'), name='vz_sel'), row=2, col=1)
    fig.add_trace(go.Scatter(x=t, y=d['fdm_gamma_target_rad'], mode='lines', line=dict(color='royalblue', width=1, dash='dot'), name='v2 gamma target'), row=6, col=1)

    fig.add_trace(go.Scatter(x=t, y=d['CAS'], mode='lines', line=dict(color='gray', width=1), name='observed CAS'), row=3, col=1)
    m = d['cas_sel'].notna()
    fig.add_trace(go.Scatter(x=t[m], y=d.loc[m, 'cas_sel'], mode='lines', line=dict(color='red', width=2, shape='hv'), name='cas_sel'), row=3, col=1)
    fig.add_trace(go.Scatter(x=t, y=d['fdm_cas_sel_kt'], mode='lines', line=dict(color='royalblue', width=1.4), name='v2 CAS propagated'), row=3, col=1)

    fig.add_trace(go.Scatter(x=t, y=d['Mach'], mode='lines', line=dict(color='gray', width=1), name='observed Mach'), row=4, col=1)
    m = d['mach_sel'].notna()
    fig.add_trace(go.Scatter(x=t[m], y=d.loc[m, 'mach_sel'], mode='lines', line=dict(color='red', width=2, shape='hv'), name='mach_sel'), row=4, col=1)
    fig.add_trace(go.Scatter(x=t, y=d['fdm_mach_sel'], mode='lines', line=dict(color='royalblue', width=1.4), name='v2 Mach propagated'), row=4, col=1)

    fig.add_trace(go.Scatter(x=t, y=d['observed_tas_kt'], mode='lines', line=dict(color='gray', width=1), name='observed TAS'), row=5, col=1)
    fig.add_trace(go.Scatter(x=t, y=d['fdm_tas_target_kt'], mode='lines', line=dict(color='royalblue', width=1.4, dash='dot'), name='v2 TAS target'), row=5, col=1)
    fig.add_trace(go.Scatter(x=t, y=d['tas_intent_kt'], mode='lines', line=dict(color='red', width=2), name='TAS from thesis commands'), row=5, col=1)

    fig.add_trace(go.Scatter(x=t, y=d['observed_gamma_rad'], mode='lines', line=dict(color='gray', width=1), name='observed gamma'), row=6, col=1)
    fig.add_trace(go.Scatter(x=t, y=d['gamma_intent_rad'], mode='lines', line=dict(color='red', width=2), name='gamma from thesis commands'), row=6, col=1)

    fig.update_layout(height=1450, title_text=title, showlegend=True, hovermode='x unified')
    fig.update_yaxes(title_text='ft', row=1, col=1)
    fig.update_yaxes(title_text='fpm', row=2, col=1)
    fig.update_yaxes(title_text='kt', row=3, col=1)
    fig.update_yaxes(title_text='Mach', row=4, col=1)
    fig.update_yaxes(title_text='kt', row=5, col=1)
    fig.update_yaxes(title_text='rad', row=6, col=1)
    fig.update_xaxes(title_text='time [s]', row=6, col=1)
    display(fig)


output = widgets.Output()
prev_button = widgets.Button(description='← Prev')
next_button = widgets.Button(description='Next →')
index_label = widgets.HTML()


def render_current():
    with output:
        clear_output(wait=True)
        global flight_id, df
        flight_id = FLIGHT_IDS[FLIGHT_INDEX]
        obs = load_observed(flight_id)
        df = infer_commands(obs, CONFIG)
        index_label.value = f'<b>{FLIGHT_INDEX + 1}</b> / {len(FLIGHT_IDS)}: {flight_id}'
        print('flight:', flight_id)
        print('seconds:', int(df['t_s'].max()), '| rows:', len(df))
        for c in ['mach_sel', 'cas_sel', 'vz_sel', 'h_sel', 'tas_intent_kt', 'gamma_intent_rad']:
            n = df[c].notna().sum()
            print(f'  {c}: {n} samples ({100 * n / len(df):.1f}%)')
        plot_flight(df, flight_id)


def go_prev(_):
    global FLIGHT_INDEX
    FLIGHT_INDEX = (FLIGHT_INDEX - 1) % len(FLIGHT_IDS)
    render_current()


def go_next(_):
    global FLIGHT_INDEX
    FLIGHT_INDEX = (FLIGHT_INDEX + 1) % len(FLIGHT_IDS)
    render_current()


prev_button.on_click(go_prev)
next_button.on_click(go_next)
display(widgets.HBox([prev_button, next_button, index_label]))
display(output)
render_current()

Output()